In [58]:
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.feature_selection import mutual_info_classif
from catboost import CatBoostClassifier

In [59]:
path_data = r'..\Data\result\03_Exploratory_Data_Analysis_Test.csv'

In [60]:
df_test = pd.read_csv(path_data)

In [61]:
df_test.head(2)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,146435.0,-0.479716,0.575929,-0.150790,0.050136,1.780725,0.044582,1.196532,0.041638,-0.701079,...,0.001164,-0.141912,-0.594947,-1.486584,1.528421,-0.275132,-0.158246,-0.155639,39.60,0
1,146435.0,-0.802180,0.686047,-2.098744,-0.002076,2.512532,3.895537,1.146119,1.164420,-1.058840,...,0.269243,0.487273,0.282189,0.692181,0.185759,-0.361360,0.258838,0.188994,216.38,0


# Lakukan Transformasi Data yang sama saat olah data Train

In [62]:
df_test['Amount_Bucket'] = np.where(df_test['Amount'] < 260, '< 260',
                                        np.where(df_test['Amount'] > 2000, '> 2000', '261 - 2000'))

## Standarisasi Amount

In [63]:
scaler = joblib.load('../Output/Model/model_standard_scaler.pkl')

df_test['Amount'] = scaler.transform(df_test[['Amount']])

C:\Anaconda3\envs\pycaret4-env\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Feature Engineering

In [64]:
df_test['Bucket_V14'] = np.where(df_test['V14'] < -17.23, '< -17.23', '>= -17.23')
df_test['Bucket_V10'] = np.where(df_test['V10'] < -14.92, '< -14.92', '>= -14.92')
df_test['Bucket_V12'] = np.where(df_test['V12'] < -17.23, '< -17.23', '>= -17.23')
df_test['Bucket_V11'] = np.where(df_test['V11'] > 10.19, '> 10.19', '<= 10.19')
df_test['Bucket_V3'] = np.where(df_test['V3'] < -17.71, '< -17.71', '>= -17.71')
df_test['Bucket_V9'] = np.where(df_test['V9'] < -8.09, '< -8.09', '>= -8.09')
df_test['Bucket_V7'] = np.where(df_test['V7'] < -17.68, '< -17.68', '>= -17.68')
df_test['Bucket_V1'] = np.where(df_test['V1'] < -13.68, '< -13.68', '>= -13.68')
df_test['Bucket_V26'] = np.where(df_test['V26'] > 1.247, '> 1.247', '<= 1.247')
df_test['Bucket_Amount'] = np.where(df_test['Amount'] > 0.683, '> 0.683', '<= 0.683')
df_test['Bucket_V24'] = np.where(df_test['V24'] < -1.511, '< -1.511', '>= -1.511')
df_test['Bucket_V22'] = np.where(df_test['V22'] < -2.232, '< -2.232', 
                                     np.where(df_test['V22'] > 2.152, '> 2.152', '-2.232 s.d 2.152'))
df_test['Bucket_V8'] = np.where(df_test['V8'] < -2.957, '< -2.957', 
                                     np.where(df_test['V22'] > 4.636, '> 4.636', '-2.957 s.d 4.636'))

## Reduce Feature

In [65]:
df_test = df_test[['Bucket_Amount', 'Bucket_V14', 'Bucket_V9', 'Bucket_V7', 'Bucket_V12', 'Bucket_V1', 'Bucket_V10', 'Bucket_V24', 'Bucket_V3', 'V14', 'V17', 'V12', 'V11', 'V16', 'V10', 'V4', 'V3', 'V18', 'V9', 'V26', 'V1', 'V8', 'Amount', 'V15', 'Time', 'V20', 'Class']]

## Encoding Categorical

In [66]:
encoder = joblib.load(r'..\Output\Model\model_encoder_ordinal.pkl')

In [67]:
df_test[df_test.select_dtypes(include=['str']).columns.tolist()] = encoder.transform(df_test[df_test.select_dtypes(include=['str']).columns.tolist()])

In [68]:
df_test = df_test.drop_duplicates()

# Prepare Data

In [69]:
X_test = df_test.drop(columns='Class')
y_test = df_test['Class']

# Testing Model Ke Data Baru

In [70]:
etc_model_v3 = joblib.load("../Output/Model/model_v3_extra_trees.pkl")
rfc_model_v3 = joblib.load("../Output/Model/model_v3_random_forest.pkl")

cb_model_v3 = CatBoostClassifier()
cb_model_v3.load_model("../Output/Model/model_v3_catboost.cbm")

CatBoostClassifier(depth=9, iterations=594, l2_leaf_reg=1.119943854, learning_rate=0.01063991562, loss_function='Logloss', random_strength=0.8875933831, verbose=0)

## Extra Trees Classifier

In [71]:
start_etc_test = time.perf_counter()

etc_pred = etc_model_v3.predict_proba(X_test)[:, 1]

end_etc_test = time.perf_counter()

test_etc_model_time = end_etc_test - start_etc_test

etc_v3_auprc = average_precision_score(y_test, etc_pred)

In [72]:
print(f"Test (Inference) time Extra Trees Classifier Model \t: {test_etc_model_time:.4f} seconds")

print()
print('Score AUPRC Extra Trees Classifier Model \t:',  etc_v3_auprc)

Test (Inference) time Extra Trees Classifier Model 	: 0.2037 seconds

Score AUPRC Extra Trees Classifier Model 	: 0.9465641915443885


## Random Forest Classifier

In [73]:
start_rfc_test = time.perf_counter()

rfc_pred = rfc_model_v3.predict_proba(X_test)[:, 1]

end_rfc_test = time.perf_counter()

test_rfc_model_time = end_rfc_test - start_rfc_test

rfc_v3_auprc = average_precision_score(y_test, rfc_pred)

In [74]:
print(f"Test (Inference) time Random Forest Classifier Model: {test_rfc_model_time:.4f} seconds")

print()
print('Score AUPRC Random Forest Classifier Model \t:',  rfc_v3_auprc)

Test (Inference) time Random Forest Classifier Model: 0.4218 seconds

Score AUPRC Random Forest Classifier Model 	: 0.9976005634171908


## CatBoost Classifier

In [75]:
start_cb_test = time.perf_counter()

cb_pred = cb_model_v3.predict_proba(X_test)[:, 1]

end_cb_test = time.perf_counter()

test_cb_model_time = end_cb_test - start_cb_test

cb_v3_auprc = average_precision_score(y_test, cb_pred)

In [76]:
print(f"Test (Inference) time CatBoost Classifier Model: {test_cb_model_time:.4f} seconds")

print()
print('Score AUPRC CatBoost Classifier Model \t:',  cb_v3_auprc)

Test (Inference) time CatBoost Classifier Model: 0.0219 seconds

Score AUPRC CatBoost Classifier Model 	: 0.9441028273768789


Berikut merupakan perbandingan Performa dan Detail model V1 vs V2

<table>
<tr>
<td align="center" valign="top">

<h3>TABLE TRAIN</h3>

<table>
<tr>
<th rowspan="2">Model</th>
<th colspan="3">AUPRC</th>
<th colspan="3">Training Time</th>
<th colspan="3">Validation Time</th>
</tr>

<tr>
<th>V1</th>
<th>V2</th>
<th>V3</th>
<th>V1</th>
<th>V2</th>
<th>V3</th>
<th>V1</th>
<th>V2</th>
<th>V3</th>
</tr>

<tr>
<td>Extra Trees</td>
<td>0.81</td>
<td>0.87</td>
<td>0.87</td>
<td>30.5s</td>
<td>18.5s</td>
<td>15.6s</td>
<td>0.74s</td>
<td>0.2s</td>
<td>0.2s</td>
</tr>

<tr>
<td>Random Forest</td>
<td>0.82</td>
<td>0.89</td>
<td>0.89</td>
<td>226.5s</td>
<td>181.9s</td>
<td>110.0s</td>
<td>0.46s</td>
<td>1.0s</td>
<td>0.38s</td>
</tr>

<tr>
<td>CatBoost</td>
<td>0.81</td>
<td>0.89</td>
<td>0.89</td>
<td>29.8s</td>
<td>144.2s</td>
<td>24.6s</td>
<td>0.05s</td>
<td>0.04s</td>
<td>0.03s</td>
</tr>
</table>

</td>

<td width="40"></td>

<td align="center" valign="top">

<h3>TABLE TESTING</h3>

<table>
<tr>
<th>Model</th>
<th>AUPRC</th>
<th>Testing Time</th>
</tr>

<tr>
<td>Extra Trees</td>
<td>0.94</td>
<td>0.2s</td>
</tr>

<tr>
<td>Random Forest</td>
<td>0.99</td>
<td>0.4s</td>
</tr>

<tr>
<td>CatBoost</td>
<td>0.94</td>
<td>0.02s</td>
</tr>
</table>

</td>
</tr>
</table>

Berdasarkan table diatas, di rekomendasikan melakukan reduksi feature serta menggunakan model CatBoost Classifier. Hal ini dikarenakan:
- Meskipun memiliki performa lebih rendah daripada Random Forest Classifier, Model masih dapat dipercaya dikarenakan untuk performa 0.99 pada Random Forest bisa diakibatkan oleh Overfitting
- Dengan total data 25 ribuan, waktu yang dibutuhkan untuk melakukan Test pada data yang belum pernah dilihat sebelumnya hanya membutuhkan waktu 0.02 detik

In [77]:
df_test.shape

(25535, 27)